# vLLM-Hook Minimal Parity: Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/vllm-hook-mlx/tests/parity_tests/colab_minimal_parity.ipynb)

This notebook runs the Colab/non-Metal half of `tests/parity_tests/run_all_minimal_parity.py` directly inside a Colab GPU runtime. It is intended as a manual replacement while the local `colab` CLI workflow is unavailable.

Run the Mac/Metal side locally with the same `BENCHMARK_PREFIX`, then run this notebook on Colab with a GPU runtime. Matching runs use W&B groups named `<BENCHMARK_PREFIX>-<experiment>`.

## 1. Bootstrap Repo And Dependencies

Set `REPO_URL` and `REPO_BRANCH` to a branch that contains `tests/parity_tests/minimal_parity_benchmarks.py`. On Colab, this cell clones or refreshes the repo, installs requirements, installs the plugin package editable, and verifies CUDA is available.

In [1]:
from pathlib import Path
import importlib
import importlib.metadata as md
import importlib.util
import inspect
import json
import os
import re
import shutil
import site
import subprocess
import sys

REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "vllm-hook-mlx")
WORKDIR = Path(os.environ.get("VLLM_HOOK_COLAB_WORKDIR", "/content/vLLM-Hook"))
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.14,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_parity_binary_deps_restarted")
IN_COLAB = "google.colab" in sys.modules


def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("+ " + " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        tail = tail[-160:]
    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )


def run_capture(cmd):
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)


def norm(name):
    return name.lower().replace("_", "-")


def package_from_req_line(line: str) -> str:
    stripped = line.strip()
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())


def repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return origin_url == expected_remote


def prepare_repo(repo_url: str, branch: str, workdir: Path) -> Path:
    if not repo_url:
        if workdir.exists():
            return workdir
        raise FileNotFoundError(f"Repo URL is empty and {workdir} does not exist.")
    expected_remote = repo_url.removesuffix(".git")
    if not workdir.exists():
        run(["git", "clone", "--branch", branch, repo_url, workdir])
    elif not repo_remote_matches(workdir, expected_remote):
        shutil.rmtree(workdir)
        run(["git", "clone", "--branch", branch, repo_url, workdir])
    run(["git", "-C", workdir, "fetch", "origin", branch])
    run(["git", "-C", workdir, "checkout", branch])
    run(["git", "-C", workdir, "pull", "--ff-only", "origin", branch])
    return workdir


def assert_cuda_runtime():
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError("Choose GPU runtime and rerun from a fresh runtime.")


def package_name(node):
    if "package" in node:
        return norm(node["package"]["package_name"])
    return norm(node["package_name"])


def runtime_dependency_check():
    roots = {
        "vllm",
        "torch",
        "torchvision",
        "torchaudio",
        "numpy",
        "scipy",
        "vllm-hook-plugins",
    }

    run([sys.executable, "-m", "pip", "install", "pipdeptree"])

    tree_result = run_capture([sys.executable, "-m", "pipdeptree", "--json-tree"])
    if tree_result.returncode:
        print(tree_result.stdout)
        print(tree_result.stderr)
        raise RuntimeError("pipdeptree failed")

    tree = json.loads(tree_result.stdout)
    children = {}

    def walk(node):
        name = package_name(node)
        deps = [package_name(dep) for dep in node.get("dependencies", [])]
        children[name] = deps
        for dep in node.get("dependencies", []):
            walk(dep)

    for node in tree:
        walk(node)

    closure = set()
    stack = [norm(root) for root in roots]

    while stack:
        pkg = stack.pop()
        if pkg in closure:
            continue
        closure.add(pkg)
        stack.extend(children.get(pkg, []))

    print("\nRuntime dependency boundary:")
    for pkg in sorted(closure):
        print(" ", pkg)

    check_result = run_capture([sys.executable, "-m", "pip", "check"])
    conflict_lines = check_result.stdout.splitlines()

    hard_fail = []
    warnings = []
    pattern = re.compile(
        r"^(?P<owner>[A-Za-z0-9_.-]+)\s+.+? requirement (?P<required>[A-Za-z0-9_.-]+)"
    )

    for line in conflict_lines:
        match = pattern.search(line)
        if not match:
            warnings.append(line)
            continue
        owner = norm(match.group("owner"))
        required = norm(match.group("required"))
        if owner in closure:
            hard_fail.append(line)
        elif required in closure:
            warnings.append(line)

    print("\nDependency conflicts inside runtime boundary:")
    if hard_fail:
        for line in hard_fail:
            print(line)
        raise RuntimeError("Runtime dependency boundary has conflicts")
    print("None")

    print("\nExternal packages conflicting with shared deps:")
    if warnings:
        for line in warnings:
            print(line)
    else:
        print("None")


def import_and_contract_check():
    packages = [
        "torch",
        "vllm",
        "transformers",
        "tokenizers",
        "safetensors",
        "numpy",
        "scipy",
        "protobuf",
        "vllm-hook-plugins",
    ]

    modules = [
        "torch",
        "vllm",
        "vllm.engine.arg_utils",
        "vllm.inputs",
        "vllm_hook_plugins",
        "vllm_hook_plugins.hook_llm",
        "vllm_hook_plugins.workers.probe_hookqk_worker",
        "vllm_hook_plugins.workers.probe_hidden_states_worker",
        "vllm_hook_plugins.workers.steer_activation_worker",
    ]

    print("\nInstalled package versions:")
    for package in packages:
        try:
            print(f"{package}: {md.version(package)}")
        except md.PackageNotFoundError:
            print(f"{package}: NOT INSTALLED")

    print("\nImport/module locations:")
    for module_name in modules:
        module = importlib.import_module(module_name)
        print(f"{module_name}: OK {getattr(module, '__file__', '<namespace>')}")

    from vllm.engine.arg_utils import EngineArgs

    fields = getattr(EngineArgs, "__dataclass_fields__", {})
    params = inspect.signature(EngineArgs).parameters
    has_worker_extension = "worker_extension_cls" in fields or "worker_extension_cls" in params

    print("\nCore vLLM-Hook integration contracts:")
    print("EngineArgs.worker_extension_cls:", "OK" if has_worker_extension else "FAIL")

    if not has_worker_extension:
        raise RuntimeError("Installed vLLM does not support worker_extension_cls")


PROJECT_ROOT = prepare_repo(REPO_URL, REPO_BRANCH, WORKDIR)
os.chdir(PROJECT_ROOT)
assert_cuda_runtime()

plugin_dir = PROJECT_ROOT / "vllm_hook_plugins"
req = PROJECT_ROOT / "requirement.txt"
filtered_req = Path("/tmp/vllm_hook_colab_requirements.txt")

if "vllm" in sys.modules:
    raise RuntimeError("vllm is already imported. Restart runtime and rerun from top.")

run([sys.executable, "-m", "pip", "install", "-U", "pip"])

if req.exists():
    keep = []
    blocked = {
        "vllm",
        "torch",
        "torchvision",
        "torchaudio",
        "numpy",
        "scipy",
        "protobuf",
    }
    for line in req.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue
        package = package_from_req_line(stripped)
        if package in blocked:
            print("Skipping requirement managed by parity bootstrap:", line)
            continue
        keep.append(line)

    filtered_req.write_text("\n".join(keep) + "\n")
    run([sys.executable, "-m", "pip", "install", "-r", str(filtered_req)])

run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])
run([sys.executable, "-m", "pip", "install", "wandb", "weave", "pytest"])

if COLAB_INSTALL_VLLM:
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    # Colab currently does not support CUDA 13 on all GPU runtimes, so use CUDA 12.x wheels.
    run([
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "vllm",
        "torch",
        "torchvision",
        "torchaudio",
        "numpy",
        "scipy",
    ])

    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        for leftover in [
            site_path / "vllm",
            *site_path.glob("vllm-*.dist-info"),
            site_path / "torch",
            *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision",
            *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio",
            *site_path.glob("torchaudio-*.dist-info"),
            site_path / "numpy",
            *site_path.glob("numpy-*.dist-info"),
            site_path / "scipy",
            *site_path.glob("scipy-*.dist-info"),
        ]:
            if leftover.exists():
                print("Removing leftover:", leftover)
                shutil.rmtree(leftover) if leftover.is_dir() else leftover.unlink()

    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv",
        "pip",
        "install",
        "--system",
        "--reinstall",
        "--no-cache",
        VLLM_SPEC,
        "torch",
        "torchvision",
        "torchaudio",
        "numpy",
        "scipy",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

run([
    sys.executable,
    "-c",
    "import torch, vllm; print('torch', torch.__version__); print('vllm', getattr(vllm, '__version__', 'unknown'))",
])

run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(plugin_dir)])

plugin_src_dir = str(plugin_dir.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("Restarting Colab runtime once so Python reloads replaced binary packages.")
    os.kill(os.getpid(), 9)

runtime_dependency_check()
import_and_contract_check()

RUNNER = PROJECT_ROOT / "tests" / "parity_tests" / "minimal_parity_benchmarks.py"

print("Bootstrap complete:", PROJECT_ROOT)
print("Runner:", RUNNER)


+ git clone --branch vllm-hook-mlx https://github.com/tburleyinfo/vLLM-Hook.git /content/vLLM-Hook
Cloning into '/content/vLLM-Hook'...
+ git -C /content/vLLM-Hook fetch origin vllm-hook-mlx
From https://github.com/tburleyinfo/vLLM-Hook
 * branch            vllm-hook-mlx -> FETCH_HEAD
+ git -C /content/vLLM-Hook checkout vllm-hook-mlx
Already on 'vllm-hook-mlx'
Your branch is up to date with 'origin/vllm-hook-mlx'.
+ git -C /content/vLLM-Hook pull --ff-only origin vllm-hook-mlx
From https://github.com/tburleyinfo/vLLM-Hook
 * branch            vllm-hook-mlx -> FETCH_HEAD
Already up to date.
+ /usr/bin/python3 -m pip install -U pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Skipping requirement managed by parity bootstrap: vllm>=0.5
Skipping requirement managed by parity bootstrap: torch>=2.0
Skipping requirement

: 

: 

: 

In [ ]:
print("vLLM install handled in bootstrap cell; no separate reinstall step needed.")


## 2. Authenticate W&B And Hugging Face

Leave the constants blank to use environment variables, Colab Secrets, or prompts. The Hugging Face token is optional unless a selected model requires it.

In [ ]:
import getpass
import os

HARDCODED_WANDB_API_KEY = ""
HARDCODED_HF_TOKEN = ""

try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret_or_prompt(name: str, required: bool = False):
    hardcoded = {
        "WANDB_API_KEY": HARDCODED_WANDB_API_KEY,
        "HF_TOKEN": HARDCODED_HF_TOKEN,
    }.get(name)
    if hardcoded:
        os.environ[name] = hardcoded
        return hardcoded
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return value
    if required:
        value = getpass.getpass(f"{name}: ")
        os.environ[name] = value
        return value
    return None

secret_or_prompt("WANDB_API_KEY", required=True)
secret_or_prompt("HF_TOKEN", required=False)

if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

import wandb
wandb.login(key=os.environ.get("WANDB_API_KEY"), relogin=True)
print("W&B login configured")

## 3. Configure The Comparison Run

Use the same `BENCHMARK_PREFIX` as the local Mac run. The app-specific W&B projects are selected by `minimal_parity_benchmarks.py`; `WANDB_PROJECT` remains the fallback.

In [ ]:
EXPERIMENTS = ("hidden-states", "attn-tracker", "core-reranker", "steer-activation")
MODELS = {
    "hidden-states": os.environ.get("HIDDEN_STATES_MODEL", "Qwen/Qwen2.5-3B-Instruct"),
    "attn-tracker": os.environ.get("ATTN_TRACKER_MODEL", "RedHatAI/granite-3.1-2b-instruct-quantized.w4a16"),
    "core-reranker": os.environ.get("CORE_RERANKER_MODEL", "mistralai/Mistral-7B-Instruct-v0.3"),
    "steer-activation": os.environ.get("STEER_ACTIVATION_MODEL", "microsoft/Phi-3-mini-4k-instruct"),
}

BENCHMARK_PREFIX = os.environ.get("BENCHMARK_PREFIX", "minimal-parity")
WANDB_MODE = os.environ.get("WANDB_MODE", "online")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "vllm-hook-platform-parity")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
HARDWARE_LABEL = os.environ.get("VLLM_HOOK_HARDWARE_LABEL", "colab-gpu")
HARDWARE_KIND = os.environ.get("VLLM_HOOK_HARDWARE_KIND", "cuda")
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "2"))
TEMPERATURE = 0.0
TOP_P = float(os.environ.get("TOP_P", "1.0"))
GPU_MEMORY_UTILIZATION = float(os.environ.get("GPU_MEMORY_UTILIZATION", "0.8"))
MAX_MODEL_LEN = int(os.environ.get("MAX_MODEL_LEN", "2048"))
DTYPE = os.environ.get("DTYPE", "float16")

SHARED_INPUTS = {
    "experiments": EXPERIMENTS,
    "models": MODELS,
    "benchmark_prefix": BENCHMARK_PREFIX,
    "max_tokens": MAX_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "max_model_len": MAX_MODEL_LEN,
    "dtype": DTYPE,
}

print("benchmark prefix:", BENCHMARK_PREFIX)
print("wandb mode     :", WANDB_MODE)
print("wandb project  :", WANDB_PROJECT)
print("wandb entity   :", WANDB_ENTITY or "<default>")
print("hardware label :", HARDWARE_LABEL)
print("shared inputs  :", SHARED_INPUTS)

## 4. Helper To Run A Colab Experiment

This uses the checked-in minimal parity runner with `--backend non-metal`, matching `colab_minimal_parity_remote.py`.

In [ ]:
def run_colab_parity(experiment: str):
    if experiment not in EXPERIMENTS:
        raise ValueError(f"Expected one of {EXPERIMENTS}, got {experiment!r}")
    env = os.environ.copy()
    env["WANDB_MODE"] = WANDB_MODE
    env["WANDB_PROJECT"] = WANDB_PROJECT
    env.setdefault("VLLM_USE_V1", "1")
    env.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
    if WANDB_ENTITY:
        env["WANDB_ENTITY"] = WANDB_ENTITY

    benchmark_id = f"{BENCHMARK_PREFIX}-{experiment}"
    cmd = [
        sys.executable,
        str(RUNNER),
        experiment,
        "--backend", "non-metal",
        "--benchmark-id", benchmark_id,
        "--model", MODELS[experiment],
        "--wandb-mode", WANDB_MODE,
        "--wandb-project", WANDB_PROJECT,
        "--hardware-label", HARDWARE_LABEL,
        "--hardware-kind", HARDWARE_KIND,
        "--max-tokens", str(MAX_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-p", str(TOP_P),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--dtype", DTYPE,
    ]
    if WANDB_ENTITY:
        cmd.extend(["--wandb-entity", WANDB_ENTITY])
    run(cmd, cwd=PROJECT_ROOT, env=env)
    return PROJECT_ROOT / "tests" / "experiment_runs" / "minimal_parity" / experiment / benchmark_id / "non-metal"

## 5. Run One Experiment

Start with attention tracker because it uses the smaller default model.

In [ ]:
run_dir = run_colab_parity("hidden-states")

Run the remaining experiments when ready.

In [ ]:
# run_dir = run_colab_parity("attn-tracker")
# run_dir = run_colab_parity("core-reranker")
# run_dir = run_colab_parity("steer-activation")

Run all Colab/non-Metal experiments.

In [ ]:
# for experiment in EXPERIMENTS:
#     run_dir = run_colab_parity(experiment)

## 6. Inspect Local Outputs

The runner writes local JSON, CSV, manifest, and hook artifacts under `tests/experiment_runs/minimal_parity` before logging to W&B.

In [ ]:
import json

def show_local_outputs(run_dir: Path):
    print("Run dir:", run_dir)
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            print(" ", path.relative_to(run_dir), path.stat().st_size, "bytes")
    manifest_path = run_dir / "artifact_manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        print("\nManifest files:", len(manifest.get("files", [])))
        for row in manifest.get("files", []):
            if row.get("path", "").endswith((".pt", ".safetensors")):
                print(row["path"], row.get("bytes"), row.get("sha256"))

show_local_outputs(run_dir)

## 7. Optional Artifact Lookup

Use this to find the latest W&B artifact for one experiment in the app-specific project.

In [ ]:
APP_PROJECTS = {
    "hidden-states": "hiddenstates",
    "attn-tracker": "attntracker",
    "core-reranker": "corereranker",
    "steer-activation": "steering",
}

def download_latest_artifact(experiment="hidden-states"):
    api = wandb.Api()
    entity = WANDB_ENTITY or api.default_entity
    project = APP_PROJECTS.get(experiment, WANDB_PROJECT)
    runs = api.runs(f"{entity}/{project}", order="-created_at", per_page=100)
    run = next(
        r for r in runs
        if experiment in r.tags and "non-metal" in r.tags and "minimal-parity" in r.tags
    )
    artifact = next(a for a in run.logged_artifacts() if a.type == "vllm-hook-minimal-parity")
    artifact_dir = Path(artifact.download())
    print("Run:", run.name, run.url)
    print("Artifact:", artifact.name)
    print("Downloaded to:", artifact_dir)
    return artifact_dir

# artifact_dir = download_latest_artifact("hidden-states")